# Самостоятельная работа №1
### SQL и Pandas. Основы работы с данными

Вариант 10
Студент: _______

Указания к выполнению:
- Ответы на теоретические вопросы записываются в текстовых блоках в поле «Ответ».
- Практические задания выполняются в соответствующих кодовых ячейках.

### Задание 1. Теоретические вопросы по Pandas

#### 1.1. В чем практическая разница между обращением к данным через `.loc[]` и `.iloc[]` в библиотеке Pandas?
```python
val_a = df.loc[0, 'nickname']
val_b = df.iloc[0, 1]
```
По какому принципу находит элемент каждый из этих методов?

Ответ: __________

#### 1.2. Аналитик выполнил две команды:
```python
cnt = df['city'].nunique()
lst = df['city'].unique()
```
Чем принципиально различаются результаты в переменных `cnt` и `lst`? Что возвращает число, а что — массив самих названий?

Ответ: __________

#### 1.3. Что делает следующая строка кода? Для чего она может быть использована в аналитике?
```python
df["effective_hourly_rate"] = (df["total_paid"] / df["duration_hours"]).round(2)
```

Ответ: __________

#### 1.4. Зачем после группировки данных в Pandas аналитики вызывают метод `.reset_index()`?
```python
summary = df.groupby('category')['total_amount'].sum().reset_index()
```
Как изменится структура таблицы `summary` при добавлении `.reset_index()`?

Ответ: __________

In [ ]:
# Черновик

### Задание 2. Практическая работа в Pandas

#### 2.1. Импортируйте библиотеки `pandas`, `numpy`, `sqlite3` и выведите их версии.

In [ ]:
# Код задания 2.1
# ...

#### 2.2. Загрузите таблицу `tournaments.csv` и определите:
- Сколько строк содержится в этой таблице?
- Какие столбцы входят в датафрейм?
- Каков средний вступительный взнос участников (округлите до 2 знаков)?

In [ ]:
# Код задания 2.2
# ...

#### 2.3. Поиск и исправление ошибок в коде
При попытке загрузить файл `game_sessions.csv` и рассчитать метрики код завершается с ошибкой.

```python
import pandas as pd

# 1. Загрузка данных
df_data = pd.read_csv("game_sessions.csv")

# 2. Группировка
summary = df_data.groupby("game_title").agg({
    "total_paid": "sum"
    "duration_hours": "mean"
}).reset_index()

print(summary)
```

Вопросы:
1. Какая ошибка допущена при вызове `read_csv`?
2. Какая ошибка допущена при обработке или агрегации данных?

Ответ:
1. ...
2. ...

In [ ]:
# Код задания 2.3
# ...

### Задание 3. Теоретические вопросы по SQL

#### 3.1. В таблице `game_sessions` колонка `member_id` является внешним ключом (`FOREIGN KEY`), ссылающимся на таблицу `members`.
Зачем нужна эта связь и что произойдет, если попытаться добавить игровую сессию для несуществующего клиента с `member_id = 99999`?

Ответ: __________

#### 3.2. Администратор киберклуба случайно выполнил запрос `DELETE FROM game_sessions;` вместо удаления одной устаревшей сессии.
Если запрос выполнялся внутри открытой транзакции, какой командой можно отменить удаление до того, как изменения зафиксируются на диске?

Ответ: __________

#### 3.3. Анализ ошибки в SQL
Запрос для отбора сессий завершается с ошибкой:
```sql
SELECT 
    member_id, 
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE discounted_paid > 500;
```
СУБД возвращает ошибку: `no such column: discounted_paid`.

Вопросы:
1. Почему возникает эта ошибка, если столбец `discounted_paid` указан в `SELECT`?
2. Какой блок запроса выполняется раньше — `WHERE` или `SELECT`?

Ответ: __________

#### 3.4. Что вычисляет следующий SQL-запрос?
```sql
SELECT min_rank, COUNT(*) AS tourn_cnt, SUM(prize_fund) AS total_prizes FROM tournaments GROUP BY min_rank ORDER BY total_prizes DESC;
```

Ответ: __________

In [ ]:
# Черновик

### Задание 4. Работа с SQL в SQLite

#### 4.1. Создание таблиц и импорт данных
В коде ниже таблицы `bar_orders`, `game_sessions`, `members` уже сформированы в качестве примера. Допишите DDL-скрипт для оставшихся таблиц:

**Таблица `hardware_pcs`**:  
Таблица содержит целочисленный первичный ключ `pc_id` и обязательное название зоны `zone_name`.  
Инвентарный номер `pc_number` является обязательным уникальным текстовым полем.  
Почасовая ставка `hourly_rate` задается положительным вещественным числом.  
Объем памяти `ram_gb` задается положительным целым числом, а характеристики `gpu` и `cpu` хранятся в виде текста.  
Статус `status` принимает только значения 'Active', 'Maintenance' или 'Reserved'.

**Таблица `tournaments`**:  
Таблица содержит целочисленный первичный ключ `tournament_id`.  
Поля `tournament_name`, `game_title` и `start_date` являются обязательными текстовыми полями.  
Поле `min_rank` хранится в виде текста.  
Призовой фонд `prize_fund` и вступительный взнос `entry_fee` задаются неотрицательными вещественными числами.

После завершения описания схемы выполните ячейку для создания таблиц, импорта CSV-файлов и сверки количества записей.

In [ ]:
import sqlite3
import pandas as pd
import pathlib
import os

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ddl_script = '''
-- Пример
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL,
    email         TEXT,
    city          TEXT,
    registered_at TEXT,
    loyalty_tier  TEXT CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    balance       REAL CHECK (balance >= 0)
);

CREATE TABLE hardware_pcs (
    pc_id ...,
    pc_number ...,
    zone_name ...,
    hourly_rate ...,
    gpu ...,
    cpu ...,
    ram_gb ...,
    status ...
);

-- Пример
CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER REFERENCES members(member_id),
    pc_id          INTEGER REFERENCES hardware_pcs(pc_id),
    game_title     TEXT NOT NULL,
    session_date   TEXT NOT NULL,
    duration_hours REAL CHECK (duration_hours > 0),
    discount_promo INTEGER CHECK (discount_promo IN (0, 1)),
    total_paid     REAL CHECK (total_paid >= 0),
    client_rating  REAL
);

-- Пример
CREATE TABLE bar_orders (
    order_id     INTEGER PRIMARY KEY,
    member_id    INTEGER REFERENCES members(member_id),
    item_name    TEXT NOT NULL,
    category     TEXT NOT NULL,
    order_date   TEXT NOT NULL,
    quantity     INTEGER CHECK (quantity > 0),
    item_price   REAL CHECK (item_price > 0),
    total_amount REAL CHECK (total_amount >= 0)
);

CREATE TABLE tournaments (
    tournament_id ...,
    tournament_name ...,
    game_title ...,
    prize_fund ...,
    entry_fee ...,
    start_date ...,
    min_rank ...
);
'''

# 1. Создание таблиц
cursor.executescript(ddl_script)

# 2. Загрузка данных из CSV
data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path(f"../variant_10")
for table in ["members", "hardware_pcs", "game_sessions", "bar_orders", "tournaments"]:
    df_temp = pd.read_csv(data_dir / f"{table}.csv", sep=";")
    df_temp.to_sql(table, conn, if_exists="append", index=False)

# 3. Проверка количества строк
check_query = '''
SELECT 'members' AS tbl, COUNT(*) AS cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*) FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*) FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments;
'''
display(pd.read_sql_query(check_query, conn))

#### 4.2. Фильтрация и сортировка данных
Напишите SQL-запрос, выводящий топ-5 сессий по игре `Dota 2` с длительностью `duration_hours >= 2.5`, в которых не использовался промокод (`discount_promo = 0`).
Столбцы: `session_id`, `member_id`, `game_title`, `duration_hours`, `discount_promo`, `total_paid`.
Сортировка: по длительности (`duration_hours DESC`), затем по сумме (`total_paid DESC`).

In [ ]:
# Код задания 4.2
# ...

#### 4.3. Группировка и фильтрация агрегированных данных
Напишите SQL-запрос, который для каждого рейтинга клиента (`client_rating` из `game_sessions`, исключая NULL) рассчитывает:
- Число сессий (`sessions_count`);
- Выручку (`total_revenue`), округленную до 2 знаков;
- Среднюю длительность (`avg_duration`), округленную до 2 знаков.

Фильтрация: групп с числом сессий не менее 30 (`HAVING`).
Сортировка: по рейтингу DESC.

In [ ]:
# Код задания 4.3
# ...

#### 4.4. Объединение таблиц
Сформируйте отчет по клиентам из города `Санкт-Петербург`, объединив `members` и `game_sessions`.
Столбцы: `member_id`, `nickname`, `sessions_count`, `total_spent` (потрачено на игры, до 2 знаков).
Сортировка: по сумме (`total_spent DESC`), топ-8 строк.

In [ ]:
# Код задания 4.4
# ...

#### 4.5. Подзапросы
Напишите SQL-запрос, выводящий компьютеры клуба, объем оперативной памяти которых (`ram_gb`) выше среднего объема оперативной памяти по всему клубу.
Столбцы: `pc_id`, `pc_number`, `zone_name`, `hourly_rate`, `ram_gb`.
Сортировка: по оперативной памяти (`ram_gb DESC`), затем по номеру ПК (`pc_number ASC`).

In [ ]:
# Код задания 4.5
# ...

### Задание 5. Поиск и исправление ошибок в SQL-запросах

#### 5.1. Крупные заказы в баре клуба
Стажер пытался сопоставить заказы бара с именами клиентов:

```sql
SELECT 
    m.nickname,
    m.city,
    b.item_name,
    b.total_amount
FROM members m
JOIN bar_orders b ON m.member_id = b.order_id
WHERE b.total_amount > 400
ORDER BY b.total_amount DESC;
```

Ответ:
1. ...

In [ ]:
# Код задания 5.1
# ...

#### 5.2. Суммарные заказы клиентов в баре
Аналитик формировал список посетителей с высокими чеками в баре:

```sql
SELECT 
    m.member_id,
    m.nickname,
    SUM(b.total_amount) AS bar_spent
FROM members m
JOIN bar_orders b ON m.member_id = b.order_id
GROUP BY m.member_id
HAVING SUM(b.total_amount) > 1000
ORDER BY bar_spent DESC
LIMIT 10;
```

Ответ:
1. ...
2. ...

In [ ]:
# Код задания 5.2
# ...

#### 5.3. Сравнение призовых фондов турниров со средним фондом
Ассистент пытался найти группы турниров с фондом выше среднего по всем турнирам:

```sql
SELECT 
    game_title,
    min_rank,
    ROUND(AVG(prize_fund), 2) AS avg_prize
FROM tournaments
GROUP BY game_title
WHERE avg_prize > (
    SELECT AVG(prize_fund) 
    FROM tournaments 
    GROUP BY game_title
)
ORDER BY avg_prize DESC;
```

Ответ:
1. ...
2. ...
3. ...

In [ ]:
# Код задания 5.3
# ...